# Step 4：原因假设与Step 5：数据验证

## 三个待验证假设

- **H1：**10月7日后大量低留存来源4用户进入，是整体留存下降的主要结构性解释。
- **H2：**来源4用户首周持续使用深度较弱，与低留存表现相关。
- **H3：**来源4用户交易、付费和自动续费结构较弱，说明其订阅质量与其他主要来源存在明显差异。

分析只确认分组差异和描述性结构贡献，不作严格因果表述，也不为来源4编造真实渠道名称。


In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
run_started=time.perf_counter(); project_dir=Path.cwd().resolve()
if project_dir.name=='notebooks': project_dir=project_dir.parent
raw_dir=project_dir/'data'/'raw'; processed_dir=project_dir/'data'/'processed'; CHUNK_SIZE=1_000_000
raw_files=sorted(raw_dir.glob('*.csv')); raw_before={p.name:(p.stat().st_size,p.stat().st_mtime_ns) for p in raw_files}
base=pd.read_csv(processed_dir/'new_user_anomaly_base.csv',parse_dates=['registration_date','first_active_date'],dtype={'msno':'object','registered_via':'Int16'})
base['analysis_period']=np.select([base.registration_date.between('2015-07-01','2015-09-30'),base.registration_date.between('2015-10-01','2015-10-06'),base.registration_date.between('2015-10-07','2015-10-31')],['baseline','oct_01_06','oct_07_31'],default='other')
major_sources=[3,4,7,9]; target=base[base.registration_date.between('2015-10-07','2015-10-31')].copy(); print(f'Base rows={len(base):,}; target users={len(target):,}')

Base rows=628,250; target users=213,113


## 辛普森悖论与来源结构效应分解

整体 D30 异动经过数据质量验证后，继续沿 `registered_via` 下钻。主比较周期固定为：

- **Baseline：**2015-07-01 至 2015-09-30
- **October：**2015-10-01 至 2015-10-31

10月1–6日与10月7–31日只用于后续断点诊断，不替代本节的 Baseline vs October 主比较。

分析顺序是：整体异动 → Source 分组对比 → 聚合趋势反转识别 → Source Mix 分解 → Source 4 结构解释。反事实口径为：固定 Baseline 各 Source 的 D30，仅替换为 October Source 权重。

$$R_{cf}=\sum_s w_{Oct,s}R_{Base,s}$$

$$Mix=R_{cf}-R_{Base},\quad Within=R_{Oct}-R_{cf},\quad Total=R_{Oct}-R_{Base}$$

本节检验的是统计聚合反转与 composition effect，不证明具体营销、产品或渠道动作的因果关系。


In [2]:
# Baseline vs October: recompute every result from the cleaned user-level base.
baseline_users = base[base["registration_date"].between("2015-07-01", "2015-09-30")].copy()
october_users = base[base["registration_date"].between("2015-10-01", "2015-10-31")].copy()

assert baseline_users["msno"].is_unique and october_users["msno"].is_unique
assert baseline_users["registered_via"].notna().all() and october_users["registered_via"].notna().all()

source_baseline = baseline_users.groupby("registered_via").agg(
    Baseline_Users=("msno", "size"), Baseline_D30=("d30_retained", "mean")
)
source_october = october_users.groupby("registered_via").agg(
    Oct_Users=("msno", "size"), Oct_D30=("d30_retained", "mean")
)

source_comparison = source_baseline.join(source_october, how="outer").fillna(
    {"Baseline_Users": 0, "Oct_Users": 0}
)
source_comparison[["Baseline_Users", "Oct_Users"]] = source_comparison[
    ["Baseline_Users", "Oct_Users"]
].astype("int64")
source_comparison["Baseline_Share"] = source_comparison["Baseline_Users"] / len(baseline_users)
source_comparison["Oct_Share"] = source_comparison["Oct_Users"] / len(october_users)
source_comparison["D30_Change_pp"] = (
    source_comparison["Oct_D30"] - source_comparison["Baseline_D30"]
) * 100

baseline_overall = baseline_users["d30_retained"].mean()
october_overall = october_users["d30_retained"].mean()
total_change = october_overall - baseline_overall

overall_row = pd.DataFrame({
    "Baseline_Users": [len(baseline_users)],
    "Oct_Users": [len(october_users)],
    "Baseline_D30": [baseline_overall],
    "Oct_D30": [october_overall],
    "Baseline_Share": [1.0],
    "Oct_Share": [1.0],
    "D30_Change_pp": [total_change * 100],
}, index=pd.Index(["Overall"], name="Source"))

source_comparison_display = source_comparison.rename_axis("Source")[
    ["Baseline_Users", "Oct_Users", "Baseline_Share", "Oct_Share", "Baseline_D30", "Oct_D30", "D30_Change_pp"]
]
source_comparison_with_overall = pd.concat([source_comparison_display, overall_row])
display(source_comparison_with_overall.style.format({
    "Baseline_Users": "{:,.0f}", "Oct_Users": "{:,.0f}",
    "Baseline_Share": "{:.3%}", "Oct_Share": "{:.3%}",
    "Baseline_D30": "{:.3%}", "Oct_D30": "{:.3%}", "D30_Change_pp": "{:+.3f} pp",
}))

# Counterfactual: October source weights with Baseline source-specific D30.
assert source_comparison["Baseline_D30"].notna().all(), "October contains a Source without a Baseline D30"
counterfactual_d30 = (
    source_comparison["Oct_Share"] * source_comparison["Baseline_D30"]
).sum()
source_mix_effect = counterfactual_d30 - baseline_overall
within_source_effect = october_overall - counterfactual_d30
decomposition_residual = total_change - (source_mix_effect + within_source_effect)

decomposition_summary = pd.DataFrame({
    "Metric": ["Baseline Overall D30", "Counterfactual D30", "October Actual D30", "Source Mix Effect", "Within-Source Effect", "Total Change", "Identity Residual"],
    "Value": [baseline_overall, counterfactual_d30, october_overall, source_mix_effect, within_source_effect, total_change, decomposition_residual],
    "Unit": ["rate", "rate", "rate", "pp", "pp", "pp", "pp"],
})
display(decomposition_summary.assign(
    Display=[f"{v:.6%}" if u == "rate" else f"{v * 100:+.6f} pp" for v, u in zip(decomposition_summary.Value, decomposition_summary.Unit)]
))

major_source_change = source_comparison.loc[major_sources, "D30_Change_pp"]
all_comparable_change = source_comparison.dropna(subset=["Baseline_D30", "Oct_D30"])["D30_Change_pp"]
strict_simpson = bool(
    total_change < 0
    and major_source_change.gt(0).all()
    and all_comparable_change.ge(-1e-12).all()
)
assert strict_simpson, "The recomputed data do not satisfy the stated Simpson reversal"
h1_status = "\u652f\u6301" if strict_simpson else "\u4e0d\u652f\u6301"
assert abs(decomposition_residual) < 1e-12

# Source 4: distinguish absolute retention level from within-source direction.
source4 = source_comparison.loc[4]
source4_share_change = source4["Oct_Share"] - source4["Baseline_Share"]
source4_centered_mix = source4_share_change * (source4["Baseline_D30"] - baseline_overall)
source4_within = source4["Oct_Share"] * (source4["Oct_D30"] - source4["Baseline_D30"])
source4_summary = pd.DataFrame({
    "Metric": ["Baseline Users", "October Users", "Baseline Share", "October Share", "Share Change", "Baseline D30", "October D30", "D30 Change", "Centered Mix Contribution", "Within-Source Contribution"],
    "Value": [source4.Baseline_Users, source4.Oct_Users, source4.Baseline_Share, source4.Oct_Share, source4_share_change, source4.Baseline_D30, source4.Oct_D30, source4.Oct_D30-source4.Baseline_D30, source4_centered_mix, source4_within],
})
display(source4_summary)

print("Strict Simpson reversal:", strict_simpson)
print(f"Overall D30: {baseline_overall:.6%} -> {october_overall:.6%} ({total_change*100:+.6f} pp)")
print("Major Source D30 changes:", ", ".join(f"Source {s} {major_source_change.loc[s]:+.3f} pp" for s in major_sources))
print(f"Decomposition: Mix {source_mix_effect*100:+.6f} pp + Within {within_source_effect*100:+.6f} pp = Total {total_change*100:+.6f} pp")
print(f"Source 4: share {source4.Baseline_Share:.3%} -> {source4.Oct_Share:.3%}; D30 {source4.Baseline_D30:.3%} -> {source4.Oct_D30:.3%}")


,Baseline_Users,Oct_Users,Baseline_Share,Oct_Share,Baseline_D30,Oct_D30,D30_Change_pp
Source,,,,,,,
2,99,23,0.025%,0.010%,0.000%,0.000%,+0.000 pp
3,"187,274","20,779",47.729%,8.809%,4.184%,6.771%,+2.587 pp
4,475,"172,393",0.121%,73.084%,0.632%,2.370%,+1.739 pp
7,"56,803","25,074",14.477%,10.630%,31.905%,34.653%,+2.748 pp
8,92,30,0.023%,0.013%,0.000%,0.000%,+0.000 pp
9,"146,964","17,398",37.456%,7.376%,9.082%,15.427%,+6.345 pp
11,605,180,0.154%,0.076%,0.000%,0.000%,+0.000 pp
14,27,6,0.007%,0.003%,0.000%,0.000%,+0.000 pp
16,27,1,0.007%,0.000%,0.000%,0.000%,+0.000 pp


,Metric,Value,Unit,Display
0,Baseline Overall D30,0.100187,rate,10.018707%
1,Counterfactual D30,0.048915,rate,4.891504%
2,October Actual D30,0.071501,rate,7.150125%
3,Source Mix Effect,-0.051272,pp,-5.127203 pp
4,Within-Source Effect,0.022586,pp,+2.258621 pp
5,Total Change,-0.028686,pp,-2.868582 pp
6,Identity Residual,0.000000,pp,+0.000000 pp


,Metric,Value
0,Baseline Users,475.000000
1,October Users,172393.000000
2,Baseline Share,0.001211
3,October Share,0.730838
4,Share Change,0.729627
5,Baseline D30,0.006316
6,October D30,0.023702
7,D30 Change,0.017386
8,Centered Mix Contribution,-0.068491
9,Within-Source Contribution,0.012706


Strict Simpson reversal: True
Overall D30: 10.018707% -> 7.150125% (-2.868582 pp)
Major Source D30 changes: Source 3 +2.587 pp, Source 4 +1.739 pp, Source 7 +2.748 pp, Source 9 +6.345 pp
Decomposition: Mix -5.127203 pp + Within +2.258621 pp = Total -2.868582 pp
Source 4: share 0.121% -> 73.084%; D30 0.632% -> 2.370%


## H2：同期主要来源早期激活对比

In [3]:
def q50(s): return s.median()
h2=target[target.registered_via.isin(major_sources)].groupby('registered_via').agg(users=('msno','size'),activated_user_rate=('first_active_date',lambda s:s.notna().mean()),days_to_first_activity_mean=('days_to_first_activity','mean'),days_to_first_activity_median=('days_to_first_activity',q50),first_7d_active_days_mean=('first_7d_active_days','mean'),first_7d_active_days_median=('first_7d_active_days',q50),first_7d_total_secs_mean=('first_7d_total_secs','mean'),first_7d_total_secs_median=('first_7d_total_secs',q50),first_7d_num_100_mean=('first_7d_num_100','mean'),first_7d_num_100_median=('first_7d_num_100',q50),d7_retention=('d7_retained','mean'),d30_retention=('d30_retained','mean')).reset_index()
print(h2.to_string(index=False))
s4=h2[h2.registered_via.eq(4)].iloc[0]; others=h2[h2.registered_via.ne(4)]
lower_behavior=(s4.first_7d_active_days_mean<others.first_7d_active_days_mean.min() and s4.first_7d_total_secs_mean<others.first_7d_total_secs_mean.min() and s4.first_7d_num_100_mean<others.first_7d_num_100_mean.min())
lower_retention=s4.d7_retention<others.d7_retention.min() and s4.d30_retention<others.d30_retention.min(); weaker_activation=s4.activated_user_rate<others.activated_user_rate.min() and s4.days_to_first_activity_mean>others.days_to_first_activity_mean.max(); h2_status='支持' if lower_behavior and lower_retention and weaker_activation else ('部分支持' if lower_behavior and lower_retention else '不支持')
print(f'\nH2：{h2_status}')

 registered_via  users  activated_user_rate  days_to_first_activity_mean  days_to_first_activity_median  first_7d_active_days_mean  first_7d_active_days_median  first_7d_total_secs_mean  first_7d_total_secs_median  first_7d_num_100_mean  first_7d_num_100_median  d7_retention  d30_retention
              3  10804             0.899482                    13.458428                            0.0                   2.306275                          2.0              12805.492156                   2830.3055              43.086264                      7.0      0.149482       0.083117
              4 172259             0.906014                     5.463731                            0.0                   1.988221                          1.0               8306.931549                   1877.8130              27.166337                      4.0      0.050929       0.023708
              7  21142             0.992101                    19.697020                            2.0                   2.515

## 30天交易特征

只使用注册日至注册后30天的交易。相同用户同日多笔交易均计数；`first_*` 选择最早交易日中在原 CSV 最先出现的记录。`revenue_30d` 仅累计 `actual_amount_paid`。

In [4]:
tx_users=target[['msno','registration_date','registered_via']].reset_index(drop=True); n=len(tx_users); user_id=pd.Series(np.arange(n,dtype=np.int32),index=tx_users.msno)
calendar=pd.date_range('2015-01-01','2017-03-31'); ints=calendar.strftime('%Y%m%d').astype('int32'); date_to_ord=pd.Series(np.arange(len(calendar),dtype=np.int16),index=ints); reg_ord=tx_users.registration_date.map(pd.Series(np.arange(len(calendar),dtype=np.int16),index=calendar)).to_numpy('int16')
tx_count=np.zeros(n,dtype=np.int16); paid_count=np.zeros(n,dtype=np.int16); revenue=np.zeros(n,dtype=np.int64); any_auto=np.zeros(n,bool); any_cancel=np.zeros(n,bool)
sentinel=np.iinfo(np.int16).max; first_ord=np.full(n,sentinel,dtype=np.int16); first_plan=np.full(n,-1,dtype=np.int16); first_paid=np.full(n,-1,dtype=np.int32); first_auto=np.full(n,-1,dtype=np.int8)
rows_scanned=qualifying_rows=chunks=0

In [5]:
scan_started=time.perf_counter(); cols=['msno','payment_plan_days','actual_amount_paid','is_auto_renew','transaction_date','is_cancel']
reader=pd.read_csv(raw_dir/'transactions.csv',usecols=cols,dtype={'msno':'object','payment_plan_days':'int16','actual_amount_paid':'int32','is_auto_renew':'int8','transaction_date':'int32','is_cancel':'int8'},chunksize=CHUNK_SIZE)
for chunk in reader:
    rows_scanned+=len(chunk); chunks+=1; mapped=chunk.msno.map(user_id); valid=mapped.notna().to_numpy()
    if valid.any():
        pos=np.flatnonzero(valid); ids=mapped.to_numpy()[valid].astype('int32',copy=False); ords=chunk.transaction_date.iloc[pos].map(date_to_ord).to_numpy(); good_date=~pd.isna(ords)
        ids=ids[good_date]; pos=pos[good_date]; ords=ords[good_date].astype('int16',copy=False); rel=ords.astype('int32')-reg_ord[ids].astype('int32'); keep=(rel>=0)&(rel<=30)
        if keep.any():
            ids=ids[keep]; pos=pos[keep]; ords=ords[keep]; qualifying_rows+=len(ids); amounts=chunk.actual_amount_paid.iloc[pos].to_numpy('int32'); autos=chunk.is_auto_renew.iloc[pos].to_numpy('int8'); cancels=chunk.is_cancel.iloc[pos].to_numpy('int8'); plans=chunk.payment_plan_days.iloc[pos].to_numpy('int16')
            np.add.at(tx_count,ids,1); np.add.at(paid_count,ids,(amounts>0).astype('int16')); np.add.at(revenue,ids,amounts.astype('int64')); np.logical_or.at(any_auto,ids,autos==1); np.logical_or.at(any_cancel,ids,cancels==1)
            cand=pd.DataFrame({'id':ids,'ord':ords,'pos':np.arange(len(ids),dtype=np.int32),'plan':plans,'paid':amounts,'auto':autos}).sort_values(['id','ord','pos']).drop_duplicates('id',keep='first')
            ci=cand.id.to_numpy('int32'); earlier=cand.ord.to_numpy('int16')<first_ord[ci]
            if earlier.any(): ci=ci[earlier]; chosen=cand.iloc[np.flatnonzero(earlier)]; first_ord[ci]=chosen.ord.to_numpy('int16'); first_plan[ci]=chosen.plan.to_numpy('int16'); first_paid[ci]=chosen.paid.to_numpy('int32'); first_auto[ci]=chosen.auto.to_numpy('int8')
    if chunks%10==0: print(f'{rows_scanned:,} transaction rows scanned')
print(f'Scanned {rows_scanned:,} rows in {chunks} chunks; qualifying rows={qualifying_rows:,}; minutes={(time.perf_counter()-scan_started)/60:.2f}')

10,000,000 transaction rows scanned


20,000,000 transaction rows scanned


Scanned 21,547,746 rows in 22 chunks; qualifying rows=44,957; minutes=0.34


In [6]:
has=tx_count>0; first_dates=np.full(n,np.datetime64('NaT'),dtype='datetime64[ns]'); first_dates[has]=calendar.to_numpy()[first_ord[has]]
features=tx_users.copy(); features['has_transaction_30d']=has.astype('int8'); features['transaction_count_30d']=tx_count; features['paid_transaction_count_30d']=paid_count; features['revenue_30d']=revenue
features['first_transaction_date']=pd.to_datetime(first_dates); features['days_to_first_transaction']=(features.first_transaction_date-features.registration_date).dt.days.astype('Int64')
features['first_payment_plan_days']=pd.Series(first_plan).mask(first_plan<0).astype('Int64'); features['first_actual_amount_paid']=pd.Series(first_paid).mask(first_paid<0).astype('Int64'); features['first_is_auto_renew']=pd.Series(first_auto).mask(first_auto<0).astype('Int8')
features['any_auto_renew_30d']=any_auto.astype('int8'); features['any_cancel_30d']=any_cancel.astype('int8')
feature_path=processed_dir/'new_user_transaction_features.csv'; features.to_csv(feature_path,index=False)
assert features.msno.is_unique and len(features)==len(target) and (features.days_to_first_transaction.dropna().between(0,30)).all()
print(f'Rows={len(features):,}; unique users={features.msno.nunique():,}; size={feature_path.stat().st_size:,} bytes'); print(features.columns.tolist())

Rows=213,113; unique users=213,113; size=16,745,236 bytes
['msno', 'registration_date', 'registered_via', 'has_transaction_30d', 'transaction_count_30d', 'paid_transaction_count_30d', 'revenue_30d', 'first_transaction_date', 'days_to_first_transaction', 'first_payment_plan_days', 'first_actual_amount_paid', 'first_is_auto_renew', 'any_auto_renew_30d', 'any_cancel_30d']


## H3：同期主要来源订阅结构

In [7]:
major=features[features.registered_via.isin(major_sources)]
h3=major.groupby('registered_via').agg(users=('msno','size'),has_transaction_30d=('has_transaction_30d','mean'),avg_transaction_count_30d=('transaction_count_30d','mean'),paying_user_rate=('paid_transaction_count_30d',lambda s:(s>0).mean()),avg_revenue_30d=('revenue_30d','mean'),median_revenue_30d=('revenue_30d','median'),first_payment_plan_days_mean=('first_payment_plan_days','mean'),first_payment_plan_days_median=('first_payment_plan_days','median'),first_actual_amount_paid_mean=('first_actual_amount_paid','mean'),first_actual_amount_paid_median=('first_actual_amount_paid','median'),first_is_auto_renew_rate=('first_is_auto_renew','mean'),any_auto_renew_30d=('any_auto_renew_30d','mean'),any_cancel_30d=('any_cancel_30d','mean')).reset_index()
plan_dist=major.dropna(subset=['first_payment_plan_days']).groupby(['registered_via','first_payment_plan_days']).size().rename('users').reset_index(); plan_dist['share_within_source']=plan_dist.users/plan_dist.groupby('registered_via').users.transform('sum')
paid_dist=major.dropna(subset=['first_actual_amount_paid']).groupby(['registered_via','first_actual_amount_paid']).size().rename('users').reset_index(); paid_dist['share_within_source']=paid_dist.users/paid_dist.groupby('registered_via').users.transform('sum')
print('Subscription summary:'); print(h3.to_string(index=False)); print('\nFirst payment_plan_days distribution:'); print(plan_dist.to_string(index=False)); print('\nFirst actual_amount_paid distribution:'); print(paid_dist.to_string(index=False))
s4tx=h3[h3.registered_via.eq(4)].iloc[0]; oth=h3[h3.registered_via.ne(4)]; distinct_tx=(abs(s4tx.has_transaction_30d-oth.has_transaction_30d.mean())>.05 or abs(s4tx.any_auto_renew_30d-oth.any_auto_renew_30d.mean())>.05 or abs(s4tx.avg_revenue_30d-oth.avg_revenue_30d.mean())>10)
h3_status='部分支持' if distinct_tx else '不支持'; print(f'\nH3：{h3_status}')

Subscription summary:
 registered_via  users  has_transaction_30d  avg_transaction_count_30d  paying_user_rate  avg_revenue_30d  median_revenue_30d  first_payment_plan_days_mean  first_payment_plan_days_median  first_actual_amount_paid_mean  first_actual_amount_paid_median  first_is_auto_renew_rate  any_auto_renew_30d  any_cancel_30d
              3  10804             0.128934                   0.142262          0.120233        43.905220                 0.0                      72.01005                            30.0                     320.269921                            149.0                  0.167265            0.021936        0.000370
              4 172259             0.032294                   0.036271          0.030872        10.856820                 0.0                     68.182815                            30.0                     311.490563                            149.0                  0.153155            0.005091        0.000163
              7  21142             0

## 公开数据边界

现有数据可以确认：
- 10月7日前后 `registered_via` 结构发生巨大变化；
- 不同来源的行为和留存差异；
- 不同来源的订阅结构差异。

现有数据不能确认：
- 来源4的真实渠道名称；
- 10月7日是否上线新广告、活动、产品入口；
- `registered_via` 编码是否因业务规则调整而变化。

这些需要结合真实公司的营销、产品和埋点变更日志进一步确认。

In [8]:
raw_after={p.name:(p.stat().st_size,p.stat().st_mtime_ns) for p in raw_files}; raw_unchanged=raw_before==raw_after; runtime=time.perf_counter()-run_started
print(f'Final status: H1={h1_status}; H2={h2_status}; H3={h3_status}')
print(f'Transaction features: {feature_path}; rows={len(features):,}; bytes={feature_path.stat().st_size:,}')
print(f'Total runtime: {runtime:.2f} seconds ({runtime/60:.2f} minutes); raw unchanged={raw_unchanged}')
assert raw_unchanged

Final status: H1=支持; H2=部分支持; H3=部分支持
Transaction features: C:\Users\Administrator\Desktop\kkbox_growth_analysis\data\processed\new_user_transaction_features.csv; rows=213,113; bytes=16,745,236
Total runtime: 23.29 seconds (0.39 minutes); raw unchanged=True


## 教程式结论：从异动诊断到实验验证

1. 留存监控发现 October 整体 D30 下滑；
2. 数据质量验证排除明显的日期覆盖和日志缺失问题；
3. Source 下钻显示聚合趋势与主要分组内部趋势反向；
4. Source Mix Decomposition 将净变化拆为结构效应和来源内部效应；
5. Source 4 的 D30 自身改善，但绝对水平仍低，且用户权重剧烈扩张，改变了总体构成；
6. 该结果属于描述性统计结构归因，不是具体渠道、广告或产品动作的因果证明；
7. 因此下一步不应面向所有用户平均施策，而应优先围绕 Source 4 的激活与留存问题，进入既有 A/B 实验设计。

现有 A/B 方案保持不变：Source 4、用户 ID 哈希稳定 50/50 分流、D30 Retention 主指标、+0.3 pp MDE、α=5%、Power=80%。


In [9]:
analysis_audit = pd.DataFrame({
    "Check": [
        "Overall D30 declines", "All major Source D30 rates improve",
        "No comparable Source D30 declines", "Mix + Within equals Total",
        "Source 4 D30 improves", "Source 4 remains below Baseline overall D30",
    ],
    "Passed": [
        total_change < 0, major_source_change.gt(0).all(),
        all_comparable_change.ge(-1e-12).all(), abs(decomposition_residual) < 1e-12,
        source4.Oct_D30 > source4.Baseline_D30, source4.Oct_D30 < baseline_overall,
    ],
})
display(analysis_audit)
assert analysis_audit["Passed"].all()
print("Conclusion: overall D30 declines while every major Source D30 improves; this is a strict aggregation reversal.")
print("Business interpretation: Source 4 changes the composition through low absolute retention and rapid share expansion; its own D30 does not deteriorate.")
print("Next step: retain the existing Source 4 A/B test design to evaluate an actionable intervention.")


,Check,Passed
0,Overall D30 declines,True
1,All major Source D30 rates improve,True
2,No comparable Source D30 declines,True
3,Mix + Within equals Total,True
4,Source 4 D30 improves,True
5,Source 4 remains below Baseline overall D30,True


Conclusion: overall D30 declines while every major Source D30 improves; this is a strict aggregation reversal.
Business interpretation: Source 4 changes the composition through low absolute retention and rapid share expansion; its own D30 does not deteriorate.
Next step: retain the existing Source 4 A/B test design to evaluate an actionable intervention.
